# Colab Drive 工作流诊断入口

该 Notebook 用于 Colab 冷启动后的 Drive 持久化诊断: 挂载 Google Drive, 拉取仓库代码, 将本地 outputs 中的受治理产物镜像到 Drive, 写入工作流清单, 并在同一入口中执行一次清单重载校验。

运行依赖: 不依赖任何前序结果包, 不需要 GPU, 可在主方法或 baseline 正式运行前独立执行。该入口不参与 `pilot_paper` 或 `full_paper` 的正式统计。

正式逻辑位于 `paper_workflow/colab_utils/drive_workflow.py`, Notebook 只作为远程执行入口。独立的 reload 入口已并入本 Notebook, 避免两个 Colab 入口维护同一套挂载、拉取和路径配置。

## 运行前准备

1. 确认 Google Drive 可挂载。
2. 默认 Drive 根目录为 `/content/drive/MyDrive/SLM`。
3. 该 Notebook 不加载真实模型, 只验证 Drive 持久化、工作流清单写入与清单重载。

In [ ]:
SLM_WM_PAPER_RUN_NAME = "pilot_paper"

import os
os.environ["SLM_WM_PAPER_RUN_NAME"] = SLM_WM_PAPER_RUN_NAME

from google.colab import drive

drive.mount('/content/drive')
os.environ.setdefault('SLM_WM_DRIVE_ROOT', '/content/drive/MyDrive/SLM')
os.environ.setdefault('SLM_WM_WORKFLOW_NAME', 'colab_drive_workflow')
print({
    'drive_root': os.environ['SLM_WM_DRIVE_ROOT'],
    'workflow_name': os.environ['SLM_WM_WORKFLOW_NAME'],
})


In [ ]:
import os
import subprocess
import sys

repository_url = os.environ.get('SLM_WM_REPOSITORY_URL', 'https://github.com/RICHAAARC/SLM-WM.git')
repository_ref = os.environ.get('SLM_WM_REPOSITORY_REF', 'main')
workspace_dir = os.environ.get('SLM_WM_WORKSPACE_DIR', '/content/slm_wm_repository')

if not os.path.exists(workspace_dir):
    subprocess.run(['git', 'clone', repository_url, workspace_dir], check=True)

subprocess.run(['git', 'fetch', '--all'], cwd=workspace_dir, check=True)
subprocess.run(['git', 'checkout', repository_ref], cwd=workspace_dir, check=True)
os.chdir(workspace_dir)
if workspace_dir not in sys.path:
    sys.path.insert(0, workspace_dir)
print({'workspace_dir': workspace_dir, 'repository_ref': repository_ref})


In [ ]:
import os

from paper_workflow.colab_utils.drive_workflow import run_colab_drive_workflow

summary = run_colab_drive_workflow(
    root='.',
    drive_root=os.environ.get('SLM_WM_DRIVE_ROOT', '/content/drive/MyDrive/SLM'),
    workflow_name=os.environ.get('SLM_WM_WORKFLOW_NAME', 'colab_drive_workflow'),
    perform_mount=False,
)
summary


In [ ]:
from pathlib import Path

output_dir = Path('outputs/colab_drive_workflow')
for report_path in sorted(output_dir.glob('*.json')):
    print(report_path)
    print(report_path.read_text(encoding='utf-8')[:1200])

reload_path = output_dir / 'reload_smoke_record.jsonl'
if reload_path.exists():
    print(reload_path)
    print(reload_path.read_text(encoding='utf-8'))
